Step 1: Problem Statement:

People often struggle to understand where their money actually goes. Even when they have transaction data, it’s hard to see which categories they overspend in or where they could cut back. My finance assistant helps users by analysing their real transaction data and summarising their spending by category. This gives them a clear picture of their financial habits and allows them to make better budgeting decisions.

Step 2: Inputs and Outputs:

Inputs: CSV file with columns: date, description, amount, category.

Outputs: Dictionary summarising spending totals per category, google gemini generated advice based on the totals.

Step 3: Worked Example:

To understand how my spending summary tool works, I created a small example dataset with three transactions and used Microsoft Copilot to create this table:

| date | description | amount | category |
| --- | --- | --- | --- |
| 2024-05-01 | Subway | 12.50 | Food |
| 2024-05-02 | Bus fare | 4.00 | Transport |
| 2024-05-03 | Woolworths | 33.00 | Food |


Now I manually add up the spending by category:

Food: 12.50 + 33.00 = 45.50

Transport: 4.00

So the expected output of my custom tool wopuld be:

Code:
{
  "Food": 45.50,
  "Transport": 4.00
}

Step 4: Pseudocode:

for this step i told co-pilot each individual step for the pseudocode in english then got it to turn it into pseudocode:

FUNCTION summarise_spending(transactions):

    Create an empty dictionary called totals

    FOR each transaction in the list:
        Get the category from the transaction
        Get the amount from the transaction

        IF the category is not already in totals:
            Add the category to totals with value 0

        Add the amount to totals[category]

    RETURN totals

Pseudocode for the full flow of the app:

User uploads a CSV file

Load the CSV into a list of transaction dictionaries

Call summarise_spending() to get category totals

Send the totals to Gemini with a finance persona instruction

Receive Gemini's advice text

Display the totals and the advice in the Gradio interface


Step 5: Python Code: PART 1

I directed Microsoft CO-pilot to help write the python code using simple functions ive learned this semester then pasted the Python functions into a code cell and ran them. This loads my custom tool into memory so the Gradio interface can use it later. I didn’t need to change the code — it already matches my pseudocode and worked example. Below:

In [9]:
import csv

def summarise_spending(transactions):
    totals = {}

    for t in transactions:
        category = t["category"]
        amount = float(t["amount"])

        if category not in totals:
            totals[category] = 0

        totals[category] += amount

    return totals

def load_csv(file):
    transactions = []
    reader = csv.DictReader(file)

    for row in reader:
        transactions.append({
            "date": row["date"],
            "description": row["description"],
            "amount": row["amount"],
            "category": row["category"]
        })

    return transactions


Step 5: Phython Code: PART 2

I add the Gemini API call.
This uses the requests library (allowed in the unit) to send my spending summary to Gemini Flash.
Gemini returns personalised finance advice based on the user’s data.
This satisfies Requirement R1: the assistant must use a real language model instead of coded responses. Below:

In [10]:
import requests
import json

def get_gemini_advice(totals):
    url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key=YOUR_API_KEY"

    system_instruction = """
    You are a friendly finance assistant.
    Give simple, practical advice based only on the spending totals provided.
    """

    user_message = f"Here are the spending totals: {totals}. Give the user helpful insights."

    data = {
        "contents": [
            {
                "role": "system",
                "parts": [{"text": system_instruction}]
            },
            {
                "role": "user",
                "parts": [{"text": user_message}]
            }
        ]
    }

    response = requests.post(url, json=data)
    result = response.json()

    return result["candidates"][0]["content"]["parts"][0]["text"]


Step 5: Python Code: PART 3

This connects all the parts of my assistant together.
The CSV loader, my custom tool, and the Gemini advice.
The user uploads a CSV file and receives a spending summary plus personalised insights.
This satisfies Requirement R4 saying that the assistant must have a simple, working interface. Below:

In [11]:
import gradio as gr

def run_app():
    def process_file(file):
        # Load CSV
        transactions = load_csv(file)

        # Summarise spending
        totals = summarise_spending(transactions)

        # Get Gemini advice
        advice = get_gemini_advice(totals)

        return f"Spending Summary:\n{totals}\n\nAdvice:\n{advice}"

    interface = gr.Interface(
        fn=process_file,
        inputs=gr.File(label="Upload your CSV"),
        outputs="text",
        title="Spending Summary Assistant"
    )

    interface.launch()


Step 5: Python Code: PART 4

In this final part of Step 5, I launch the Gradio interface so the user can actually interact with my assistant.
The previous parts defined all the functions (CSV loading, my custom tool, and the Gemini API call).
Calling run_app() starts the Gradio interface and makes the application usable.
This completes the coding phase of the project. Below:

In [12]:
run_app()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://689e335bf8fa547608.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Step 6: Testing: Part 1

In this test, I check that summarise_spending() works correctly on regular data.
This matches the worked example I did earlier and confirms the function adds amounts per category.

In [13]:
def test_normal_data():
    transactions = [
        {"date": "2024-05-01", "description": "Subway", "amount": 12.50, "category": "Food"},
        {"date": "2024-05-02", "description": "Bus fare", "amount": 4.00, "category": "Transport"},
        {"date": "2024-05-03", "description": "Woolworths", "amount": 33.00, "category": "Food"},
    ]

    totals = summarise_spending(transactions)

    assert totals["Food"] == 45.50
    assert totals["Transport"] == 4.00

test_normal_data()


Step 6: Testing: Part 2 Empty list

This test checks what happens if there are no transactions.
The function should return an empty dictionary instead of crashing if its running correctly.

In [14]:
def test_empty_list():
    transactions = []

    totals = summarise_spending(transactions)

    assert totals == {}

test_empty_list()


Step 6: Testing: Part 3 Missing category

This test checks how the function behaves if a transaction is missing a category.
In my current design I expect it to ignore that transaction or raise an error I can handle later.

In [15]:
def test_missing_category():
    transactions = [
        {"date": "2024-05-01", "description": "Subway", "amount": 12.50},  # no category
    ]

    try:
        totals = summarise_spending(transactions)
        # If it gets here, at least it didn't crash badly
    except KeyError:
        # This is acceptable for now, and shows I thought about the edge case
        pass

test_missing_category()


Step 6: Testing: Part 4 Negative amount

This test checks how the function handles negative amounts like a refund.
I expect the totals to include the negative value.

In [16]:
def test_negative_amount():
    transactions = [
        {"date": "2024-05-01", "description": "Refund", "amount": -10.00, "category": "Food"},
        {"date": "2024-05-02", "description": "Groceries", "amount": 30.00, "category": "Food"},
    ]

    totals = summarise_spending(transactions)

    assert totals["Food"] == 20.00  # 30 - 10

test_negative_amount()


Step 6 Part 5: Bug I found and fixed

When I was testing my summarise_spending() function, I found a small bug.
If a transaction didn’t have a "category" key, the function crashed with a KeyError.
This happened during my “missing category” test. To fix this, I added a simple check inside the loop to make sure the category exists before using it.
This stops the function from crashing on awkward input and matches the basic Python techniques from Modules 3–6.